# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure `.env` defines `OPENROUTER_API_KEY`.
- Tweak `agent_configs` / `turns_per_agent` to explore different models, prompts, and persistence paths.
- Use the CLI (`agora persona ...` or `agora run --config ...`) to automate debates outside the notebook.

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.append("../src")
load_dotenv()

from agora.workflows import run_debate_session, print_agent_histories

# Public debate
Define a lightweight two-agent setup and run the debate.

In [ ]:
turns_per_agent = 2

agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is trying to sell you a widget.'},
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]

agora, agents = run_debate_session(
    agent_configs,
    turns_per_agent=turns_per_agent,
    verbose=True,
)

print_agent_histories(agents)

# Public debate with private reflections
Demonstrates private reflections, pre/post interviews, and optional snapshots.

In [ ]:
private_turns_per_agent = 5

private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Beta', 'role': 'Beta is your potential buyer.'},
            {'name': 'Gamma', 'role': 'Gamma is observing your conversation.'}
        ],
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response': {'instruction': 'Alpha, what do you think of this interaction so far?', 'keep': True},
        'pre_interview': {'instruction': 'Alpha, in one sentence, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Alpha, after this exchange, in one sentence, summarize how it went and any next steps.', 'keep': False},
    },
    {
        'name': 'Beta',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is your salesman.'},
            {'name': 'Gamma', 'role': 'Gamma is observing your conversation.'}
        ],
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response': {'instruction': 'Beta, what do you think of this interaction so far?', 'keep': False},
        'pre_interview': {'instruction': 'Beta, in one sentence, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Beta, after this exchange, in one sentence, summarize how it went and any next steps.', 'keep': False},
    },
    {
        'name': 'Gamma',
        'model': 'openai/gpt-5-mini',
        'self_role': 'You are Gamma, listening to a conversation between a salesman and a potential buyer. You helpfully ad-lib their conversation in concise, single sentences.',
        'perceived_nonself_roles': [
            {'name': 'Alpha', 'role': 'Alpha is the salesman.'},
            {'name': 'Beta', 'role': 'Beta is the potential buyer.'}
        ],
        'response_instruction': 'Gamma, offer your next public utterance to Alpha and Beta.',
        'private_response': {'instruction': 'Gamma, what do you think of this interaction so far?', 'keep': True},
        'pre_interview': {'instruction': 'Gamma, in one sentence, before we start, what do you plan to focus on?', 'keep': False},
        'post_interview': {'instruction': 'Gamma, after this exchange, in one sentence, summarize how it went and any next steps.', 'keep': False},
    },
]

snapshot_path = Path('snapshots/reflection_snapshot.json')
load_snapshot_flag = False  # Set True to resume from a saved snapshot.
save_snapshot_flag = True  # Set True to persist the new session.
skip_first_agent_first_reflection = True

private_agora, private_agents = run_debate_session(
    private_agent_configs,
    turns_per_agent=private_turns_per_agent,
    verbose=True,
    snapshot_path=snapshot_path,
    load_snapshot_flag=load_snapshot_flag,
    save_snapshot_flag=save_snapshot_flag,
    skip_first_agent_first_reflection=skip_first_agent_first_reflection,
)

print_agent_histories(private_agents)